# StyleTTS2 Fine-tuning Guide

> This is a private/commercial notebook built by Trelis Research. Purchase a copy at Trelis.com.

This Jupyter notebook guides you through the process of fine-tuning a StyleTTS2 model for text-to-speech tasks. It covers setting up the environment, preparing the data, configuring the model, and initiating the training process.

Subscribe to Trelis Research emails to get notified each time a new video tutorial is published.

## Environment Setup
First, we set up the necessary environment by installing required packages and tools.


In [1]:
!apt-get update > /dev/null # update quietly
!apt-get install git-lfs
!pip install pandas -q
!pip install -q SoundFile torchaudio munch torch pydub pyyaml librosa nltk matplotlib accelerate transformers phonemizer einops einops-exts tqdm typing-extensions git+https://github.com/resemble-ai/monotonic_align.git
!apt-get install espeak-ng -y 
!pip install -q tensorboard peft==0.6.0

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 92 not upgraded.
Need to get 3503 kB of archives.
After this operation, 10.4 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 git-lfs amd64 3.0.2-1ubuntu0.2 [3503 kB]
Fetched 3503 kB in 1s (2457 kB/s)  
debconf: delaying package configuration, since apt-utils is not installed
Selecting previously unselected package git-lfs.
(Reading database ... 20729 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.0.2-1ubuntu0.2_amd64.deb ...
Unpacking git-lfs (3.0.2-1ubuntu0.2) ...
Setting up git-lfs (3.0.2-1ubuntu0.2) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 75.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.5/505.5 kB 139.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

## Preparing the Dataset directory

Here, we prepare the data directory and download the necessary voice dataset.
This downloads the voice dataset from Hugging Face, moves it to the Data directory, and removes the temporary download folder. This dataset is specifically curated by Trelis Research for optimal fine-tuning results.

In [13]:
!git clone https://github.com/yl4579/StyleTTS2.git
%cd StyleTTS2
!git-lfs clone https://huggingface.co/yl4579/StyleTTS2-LibriTTS
!mv StyleTTS2-LibriTTS/Models .

Cloning into 'StyleTTS2'...
remote: Enumerating objects: 372, done.
remote: Counting objects: 100% (245/245), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 372 (delta 201), reused 143 (delta 143), pack-reused 127
Receiving objects: 100% (372/372), 133.96 MiB | 32.94 MiB/s, done.
Resolving deltas: 100% (206/206), done.
Updating files: 100% (48/48), done.
/workspace/StyleTTS2
          with new flags from 'git clone'

'git clone' has been updated in upstream Git to have comparable
speeds to 'git lfs clone'.
Cloning into 'StyleTTS2-LibriTTS'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 25 (delta 0), reused 0 (delta 0), pack-reused 13 (from 1)
Unpacking objects: 100% (25/25), 3.73 KiB | 39.00 KiB/s, done.


In [14]:
!rm -rf Data/train_list.txt 
!rm -rf Data/val_list.txt 
!rm -rf Data/wavs

In [15]:
!git-lfs clone https://huggingface.co/datasets/Trelis/trelis_voice 
!mv ./trelis_voice/* ./Data/ 
!rm -rf trelis_voice

          with new flags from 'git clone'

'git clone' has been updated in upstream Git to have comparable
speeds to 'git lfs clone'.
Cloning into 'trelis_voice'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 71 (delta 0), reused 0 (delta 0), pack-reused 3 (from 1)
Unpacking objects: 100% (71/71), 30.26 KiB | 108.00 KiB/s, done.


## Configuring the Model

Now we configure the model parameters according to Trelis Research recommendations.

This loads the Trelis-specific configuration file, updates key parameters (data path, batch size, max length, and joint epoch), and saves the updated configuration. These settings are optimized for the Trelis voice dataset and typical GPU configurations. Try and experiment different configurations according to your computational resources.

In [16]:
config_path = "Configs/config_ft.yml"

import yaml
config = yaml.safe_load(open(config_path))

config['data_params']['root_path'] = "Data/wavs" # write the path to wavs directory 
config['batch_size'] = 4 # choose according to your RAM
config['max_len'] = 700 # Choose according to your RAM
config['loss_params']['joint_epoch'] = 110 # don't do SLM Adverserial training if you don't have enough ram

with open(config_path, 'w') as outfile:
  yaml.dump(config, outfile, default_flow_style=True)

In [17]:
!python train_lora_finetune.py --config_path ./Configs/config_ft.yml --lora_r 16 --lora_alpha 32 

python: can't open file '/workspace/StyleTTS2/train_lora_finetune.py': [Errno 2] No such file or directory


## Push the Trained Model To Hub

In [ ]:
!pip install huggingface_hub

from huggingface_hub import login, create_repo, upload_folder

# Log in to Hugging Face
login()

# Create a new model repository
repo_name = "your-username/your-model-name"  # Replace with your desired repository name
create_repo(repo_name, repo_type="model")

# Upload checkpoints
upload_folder(
    folder_path="./checkpoints",  # Replace with the path to your checkpoints folder
    repo_id=repo_name,
    repo_type="model",
    path_in_repo="checkpoints"
)

# Upload config files
upload_folder(
    folder_path="./config",  # Replace with the path to your config files folder
    repo_id=repo_name,
    repo_type="model",
    path_in_repo="config"
)

print(f"Checkpoints and config files uploaded successfully to {repo_name}")

## Conclusion
This Trelis Research notebook sets up the environment for fine-tuning a StyleTTS2 model using the Trelis voice dataset and initiates the training process. However, the error at the end indicates that some debugging or adjustments may be necessary before successful training can be achieved. Make sure to review the data preparation steps and model configuration if you encounter similar issues.